# Eksplorasi Transformer — MLFT Jigsaw (Multi-label Toxicity)

**Penanggung jawab:** Eksplorasi Transformer · **Mata kuliah:** Machine Learning for Text · **Dasar:** [TANGGUNG_JAWAB_DAN_ANALISIS.md](../../TANGGUNG_JAWAB_DAN_ANALISIS.md)

Notebook ini menjawab pertanyaan: **faktor Transformer mana yang benar-benar mempengaruhi deteksi toksisitas, dan seberapa yakin kita?**
Ia memperbaiki kelemahan metodologi eksplorasi sebelumnya (satu seed, perubahan tidak terisolasi, sampel kecil tanpa interval kepercayaan).

## Peta isi

| Bagian | Isi | Menjawab |
|---|---|---|
| 0 | Setup, konfigurasi, mode *smoke test* | Reproducibility |
| 1 | Data & subsampel terstratifikasi | Anti-leakage (**test tidak disentuh**) |
| 2 | Anatomi Transformer (tokenizer BPE, parameter, forward pass) | ML-T5-02 |
| 3 | Fungsi training modular (truncation, pos_weight, freeze, model) | ML-T5-01, T7 |
| 4 | Matriks eksperimen **terkontrol, multi-seed** | Catatan kritis #3, #6 |
| 5 | Agregasi: mean ± std, uji berpasangan (bootstrap) | Catatan kritis #2 |
| 6 | Analisis panjang teks: standar vs Head+Tail | Hipotesis truncation |
| 7 | Interpretabilitas: gradient×input & attention `<s>` | *Mengapa* model salah |
| 8 | Kalibrasi probabilitas (temperature scaling) | Pemilihan threshold |
| 9 | Penyimpanan hasil & template kesimpulan | Laporan |

## Hipotesis & aturan keputusan (ditetapkan SEBELUM melihat hasil)

| ID | Hipotesis | Eksperimen | Aturan |
|---|---|---|---|
| H1 | Head+Tail meningkatkan recall pada komentar **panjang** dibanding truncation standar, tanpa menurunkan Macro-F1 global | Grup A | Diterima jika selisih mean > 1 std antar-seed **dan** CI bootstrap tidak mencakup 0 |
| H2 | `pos_weight` adalah faktor dengan efek terbesar terhadap Macro-F1 | Grup B | Bandingkan efek terhadap grup lain (ukuran efek, bukan p-value) |
| H3 | Membekukan sebagian lapisan bawah menurunkan performa hanya sedikit | Grup C | Diterima jika penurunan < 1 std |
| H4 | Model lebih besar (`roberta-base`) membantu label langka | Grup D | Lihat F1 per-label langka (`threat`, `severe_toxic`, `identity_hate`) |
| H5 | `max_len` lebih besar membantu hanya jika komentar yang terpotong banyak | Grup E | Hubungkan dengan % truncation |

> [!IMPORTANT]
> **Aturan anti-leakage:** seluruh notebook hanya memakai *train* dan *validation*. Data *test* tidak dimuat sama sekali.
> Untuk klaim akhir, konfigurasi terbaik harus dibekukan lalu diuji **satu kali** pada test lewat pipeline utama (`src/evaluate_test.py`).

> [!NOTE]
> **Etika data:** komentar toksik bersifat ofensif. Bagian interpretabilitas menyensor kata kasar secara otomatis (`SHOW_RAW_TEXT = False`).

## 0. Setup & Konfigurasi

Jalankan sel instalasi **sekali** (di Colab atau environment baru). Di Colab pilih runtime **GPU (T4)**.

In [ ]:
# Hapus tanda komentar jika paket belum terpasang (Colab / environment baru)
# %pip install -q torch transformers scikit-learn iterative-stratification pandas numpy matplotlib seaborn requests

In [ ]:
import os, sys, json, time, math, random, re, copy, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
from sklearn.metrics import f1_score, precision_score, recall_score, hamming_loss

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

# ------------------------------------------------------------------
# KONFIGURASI UTAMA  (ubah di sini)
# ------------------------------------------------------------------
# SMOKE_TEST=True -> model mini acak + data kecil, hanya untuk memastikan seluruh notebook berjalan (menit, bukan jam).
SMOKE_TEST = os.environ.get("SMOKE_TEST", "0") == "1"

SEED       = 42
SEEDS      = [42, 43, 44]          # >=3 seed agar std bermakna (kritik #3)
N_TRAIN    = 1000                  # sama dengan eksperimen sebelumnya agar komparabel
N_VAL      = 300
EPOCHS     = 2
BATCH_SIZE = 16
RUN_GROUPS = ["A_trunc"]           # pilih dari: A_trunc, B_posweight, C_freeze, D_model, E_maxlen
SHOW_RAW_TEXT = False              # True = tampilkan teks asli ofensif (tidak disarankan untuk laporan)

if SMOKE_TEST:
    SEEDS, N_TRAIN, N_VAL, EPOCHS = [42, 43], 64, 48, 1
    RUN_GROUPS = ["A_trunc", "B_posweight", "C_freeze"]

LABEL_COLS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
RARE_LABELS = ["severe_toxic", "threat", "identity_hate"]
NUM_LABELS = len(LABEL_COLS)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def find_project_dir():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if p.name == "mlft-jigsaw":
            return p
        if (p / "mlft-jigsaw").is_dir():
            return p / "mlft-jigsaw"
    return Path("/content/mlft-jigsaw")   # fallback Colab

PROJECT_DIR = find_project_dir()
DATA_DIR = PROJECT_DIR / "data"
OUT_DIR = PROJECT_DIR / "experiments" / ("transformer_exploration_smoke" if SMOKE_TEST else "transformer_exploration")
for d in (DATA_DIR, OUT_DIR, OUT_DIR / "probs"):
    d.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
print(f"torch {torch.__version__} | device={DEVICE} | amp={USE_AMP} | smoke={SMOKE_TEST}")
print("PROJECT_DIR:", PROJECT_DIR)
print("OUT_DIR    :", OUT_DIR)

## 1. Data & Subsampel Terstratifikasi

* Hanya `train.csv` yang dipakai (diunduh dari mirror Hugging Face bila belum ada).
* Split train/val memakai `data/split_ids.json` bila tersedia (identik dengan pipeline utama); jika tidak, dibuat ulang dengan seed 42 (stratifikasi multi-label).
* Subsampel 1.000 train / 300 val memakai metode & seed yang sama dengan `run_experiments.py`.

> [!WARNING]
> Dengan 300 sampel validation, label langka sangat sedikit (`threat` ≈ 1, `severe_toxic` ≈ 3, `identity_hate` ≈ 3).
> Metrik per-label langka **rapuh**; gunakan sebagai petunjuk arah, bukan bukti. Naikkan `N_VAL` bila sumber daya memungkinkan.

In [ ]:
import requests

BASE_URL = "https://huggingface.co/datasets/thesofakillers/jigsaw-toxic-comment-classification-challenge/resolve/main"

def ensure_train_csv():
    dest = DATA_DIR / "train.csv"
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    print("Mengunduh train.csv ...")
    with requests.get(f"{BASE_URL}/train.csv", stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    return dest

df_full = pd.read_csv(ensure_train_csv())
print("train penuh:", df_full.shape)
assert df_full[LABEL_COLS].isin([0, 1]).all().all(), "Ada label di luar {0,1}"

try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
    HAS_ITERSTRAT = True
except ImportError:
    HAS_ITERSTRAT = False
    print("PERINGATAN: iterative-stratification tidak ada -> fallback split acak (label langka tidak terjamin)")

def stratified_take(df, size, seed):
    """Ambil `size` baris dengan stratifikasi multi-label (fallback: acak)."""
    if HAS_ITERSTRAT:
        s = MultilabelStratifiedShuffleSplit(n_splits=1, train_size=size, test_size=len(df) - size, random_state=seed)
        idx, _ = next(s.split(df["comment_text"].values, df[LABEL_COLS].values))
    else:
        idx = np.random.RandomState(seed).permutation(len(df))[:size]
    return df.iloc[idx].reset_index(drop=True)

split_file = DATA_DIR / "split_ids.json"
if split_file.exists():
    ids = json.load(open(split_file))
    assert max(ids["val_indices"]) < len(df_full) and max(ids["train_indices"]) < len(df_full)
    df_tr_full = df_full.iloc[ids["train_indices"]].reset_index(drop=True)
    df_va_full = df_full.iloc[ids["val_indices"]].reset_index(drop=True)
    print(f"split dimuat dari split_ids.json: train={len(df_tr_full)}, val={len(df_va_full)}")
else:
    if HAS_ITERSTRAT:
        s = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.10, random_state=SEED)
        tr_i, va_i = next(s.split(df_full["comment_text"].values, df_full[LABEL_COLS].values))
    else:
        perm = np.random.RandomState(SEED).permutation(len(df_full)); va_i, tr_i = perm[:int(0.1 * len(df_full))], perm[int(0.1 * len(df_full)):]
    df_tr_full, df_va_full = df_full.iloc[tr_i].reset_index(drop=True), df_full.iloc[va_i].reset_index(drop=True)
    print("split dibuat ulang (seed 42, 90/10)")

df_train = stratified_take(df_tr_full, N_TRAIN, SEED)
df_val = stratified_take(df_va_full, N_VAL, SEED)

assert len(set(df_train["id"]) & set(df_val["id"])) == 0, "KEBOCORAN: ada komentar yang sama di train & val"
dist = pd.DataFrame({"train_%": df_train[LABEL_COLS].mean() * 100, "val_%": df_val[LABEL_COLS].mean() * 100,
                     "val_n_positif": df_val[LABEL_COLS].sum()}).round(2)
display(dist)
print(f"Train={len(df_train)} | Val={len(df_val)} | komentar bersih val={(df_val[LABEL_COLS].sum(axis=1) == 0).mean():.1%}")

## 2. Anatomi Transformer

Menjawab ML-T5-02: **teks mentah → tokenizer (BPE) → `input_ids` + `attention_mask` → representasi kontekstual → 6 logit → sigmoid**.
Perbandingan ukuran model dihitung dari konfigurasi (tanpa mengunduh bobot).

In [ ]:
MODEL_SMALL, MODEL_BASE = "distilroberta-base", "roberta-base"

rows = []
for name in (MODEL_SMALL, MODEL_BASE):
    cfg = AutoConfig.from_pretrained(name, num_labels=NUM_LABELS, problem_type="multi_label_classification")
    with torch.device("meta"):
        m = AutoModelForSequenceClassification.from_config(cfg)
    total = sum(p.numel() for p in m.parameters())
    emb = sum(p.numel() for p in m.base_model.embeddings.parameters())
    head = sum(p.numel() for p in m.classifier.parameters())
    rows.append({"model": name, "layers": cfg.num_hidden_layers, "hidden": cfg.hidden_size, "heads": cfg.num_attention_heads,
                 "vocab": cfg.vocab_size, "params_total(M)": round(total / 1e6, 1), "embeddings(M)": round(emb / 1e6, 1),
                 "classifier_head(K)": round(head / 1e3, 1)})
display(pd.DataFrame(rows))
print("Catatan: head klasifikasi hanya ~0,6 juta parameter; hampir semua kapasitas ada di encoder pretrained.")

In [ ]:
tok_demo = AutoTokenizer.from_pretrained(MODEL_SMALL)

examples = ["You are doing a great job, thanks for the help!",
            "I disagree with this edit; please read the sources again.",
            "SHOUTING DOES NOT MAKE YOU RIGHT!!!"]
for s in examples:
    ids = tok_demo(s)["input_ids"]
    print(f"[{len(ids):2d} token] {tok_demo.convert_ids_to_tokens(ids)}")

print("\nSpecial tokens:", {k: v for k, v in tok_demo.special_tokens_map.items()})
print("pad_token_id =", tok_demo.pad_token_id, "| cls(<s>) =", tok_demo.cls_token_id, "| sep(</s>) =", tok_demo.sep_token_id)
print("\nPerhatikan: 'Ġ' menandai awal kata (BPE level byte); huruf KAPITAL dipecah berbeda dari huruf kecil ->")
print("pola ini menjelaskan mengapa komentar SHOUTING bisa berperilaku berbeda (kasus FP pada laporan tahap II).")

## 3. Fungsi Training Modular

Satu fungsi `run_experiment(cfg, seed)` dengan parameter yang dapat diubah **satu per satu** (prinsip eksperimen terkontrol):

| Parameter | Nilai | Arti |
|---|---|---|
| `model` | `distilroberta-base`, `roberta-base` | kapasitas model |
| `max_len` | 64 / 128 / 256 | panjang input maksimum |
| `trunc` | `standard`, `head_tail`, `tail` | strategi pemotongan teks panjang |
| `pos_weight` | `none`, `sqrt`, `full` | penanganan imbalance pada `BCEWithLogitsLoss` |
| `freeze` | 0..N | jumlah lapisan bawah yang dibekukan (+ embedding); ≥ jumlah lapisan = hanya melatih head |
| `lr`, `epochs`, `bs` | | optimasi |

Seleksi epoch memakai **Macro-F1 validation (τ=0,5)** seperti pipeline utama.
Pra-tokenisasi di-*cache* agar pemotongan murah dibandingkan antar-strategi.

In [ ]:
# ---------------------- Metrik ----------------------
def evaluate_multi_label(y_true, y_probs, threshold=0.5):
    thr = np.asarray([threshold] * NUM_LABELS) if np.isscalar(threshold) else np.asarray([threshold[c] for c in LABEL_COLS])
    y_pred = (y_probs >= thr).astype(int)
    per = {}
    for i, c in enumerate(LABEL_COLS):
        per[c] = {"P": precision_score(y_true[:, i], y_pred[:, i], zero_division=0),
                  "R": recall_score(y_true[:, i], y_pred[:, i], zero_division=0),
                  "F1": f1_score(y_true[:, i], y_pred[:, i], zero_division=0),
                  "support": int(y_true[:, i].sum())}
    return {"macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
            "macro_p": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "macro_r": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "hamming": hamming_loss(y_true, y_pred), "per_label": per}

THRESHOLDS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
def best_global_threshold(y_true, y_probs):
    scores = {t: f1_score(y_true, (y_probs >= t).astype(int), average="macro", zero_division=0) for t in THRESHOLDS}
    t = max(scores, key=scores.get)
    return t, scores[t]

# ---------------------- Tokenisasi & truncation ----------------------
_TOK_CACHE, _RAW_CACHE = {}, {}
def get_tokenizer(name):
    key = MODEL_BASE if "roberta" in name else name        # vocab distilroberta == roberta-base
    if key not in _TOK_CACHE:
        _TOK_CACHE[key] = AutoTokenizer.from_pretrained(key)
    return _TOK_CACHE[key]

def raw_ids(tok, texts, tag):
    """Token ID tanpa special token & tanpa truncation (di-cache)."""
    if tag not in _RAW_CACHE:
        _RAW_CACHE[tag] = tok([str(t) if pd.notna(t) else "" for t in texts], add_special_tokens=False, truncation=False)["input_ids"]
    return _RAW_CACHE[tag]

def truncate_ids(ids, max_len, mode):
    inner = max_len - 2                                   # sisakan <s> dan </s>
    if len(ids) > inner:
        if mode == "head_tail":
            head = inner // 2
            ids = ids[:head] + ids[-(inner - head):]
        elif mode == "tail":
            ids = ids[-inner:]
        else:                                             # standard = ambil awal
            ids = ids[:inner]
    return ids

def add_special(tok, ids):
    """Format RoBERTa: <s> ids </s> (kompatibel lintas versi transformers)."""
    return [tok.cls_token_id] + list(ids) + [tok.sep_token_id]

class EncodedDataset(torch.utils.data.Dataset):
    def __init__(self, tok, raw, labels, max_len, mode):
        self.items = [add_special(tok, truncate_ids(r, max_len, mode)) for r in raw]
        self.labels = labels.astype(np.float32)
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i], self.labels[i]

def make_collate(pad_id):
    def collate(batch):
        L = max(len(x) for x, _ in batch)
        ids = torch.full((len(batch), L), pad_id, dtype=torch.long)
        att = torch.zeros((len(batch), L), dtype=torch.long)
        for k, (x, _) in enumerate(batch):
            ids[k, :len(x)] = torch.tensor(x); att[k, :len(x)] = 1
        y = torch.tensor(np.stack([l for _, l in batch]), dtype=torch.float32)
        return ids, att, y
    return collate

# ---------------------- Model ----------------------
def create_model(name, freeze=0, eager_attn=False):
    kw = dict(num_labels=NUM_LABELS, problem_type="multi_label_classification",
              id2label=dict(enumerate(LABEL_COLS)), label2id={c: i for i, c in enumerate(LABEL_COLS)})
    extra = {"attn_implementation": "eager"} if eager_attn else {}
    if SMOKE_TEST:                                        # model mini acak: hanya untuk uji alur
        cfg = AutoConfig.from_pretrained(name, **kw)
        cfg.hidden_size, cfg.num_hidden_layers, cfg.num_attention_heads, cfg.intermediate_size = 64, 4, 2, 128
        model = AutoModelForSequenceClassification.from_config(cfg, **extra)
    else:
        model = AutoModelForSequenceClassification.from_pretrained(name, config=AutoConfig.from_pretrained(name, **kw), **extra)
    if freeze > 0:
        base = model.base_model
        for p in base.embeddings.parameters(): p.requires_grad = False
        for layer in base.encoder.layer[:freeze]:
            for p in layer.parameters(): p.requires_grad = False
    return model

def count_params(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return tr, sum(p.numel() for p in model.parameters())

def make_loss(mode, y_train):
    if mode == "none":
        return torch.nn.BCEWithLogitsLoss()
    pos = y_train.sum(0); neg = len(y_train) - pos
    ratio = neg / np.maximum(pos, 1)
    w = np.sqrt(ratio) if mode == "sqrt" else np.minimum(ratio, 100.0)   # 'full' dipotong 100 agar stabil
    return torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(w, dtype=torch.float32, device=DEVICE))

# ---------------------- Training ----------------------
@torch.no_grad()
def predict_probs(model, loader):
    model.eval(); out, ys = [], []
    for ids, att, y in loader:
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            logits = model(input_ids=ids.to(DEVICE), attention_mask=att.to(DEVICE)).logits
        out.append(torch.sigmoid(logits.float()).cpu().numpy()); ys.append(y.numpy())
    return np.vstack(out), np.vstack(ys)

BASE_CFG = dict(model=MODEL_SMALL, max_len=128, trunc="standard", lr=3e-5, epochs=EPOCHS,
                bs=BATCH_SIZE, pos_weight="sqrt", freeze=0)

def run_experiment(cfg, seed, keep_state=False):
    cfg = {**BASE_CFG, **cfg}
    set_seed(seed)
    tok = get_tokenizer(cfg["model"])
    raw_tr = raw_ids(tok, df_train["comment_text"], "train")
    raw_va = raw_ids(tok, df_val["comment_text"], "val")
    y_tr, y_va = df_train[LABEL_COLS].values, df_val[LABEL_COLS].values

    ds_tr = EncodedDataset(tok, raw_tr, y_tr, cfg["max_len"], cfg["trunc"])
    ds_va = EncodedDataset(tok, raw_va, y_va, cfg["max_len"], cfg["trunc"])
    coll = make_collate(tok.pad_token_id)
    g = torch.Generator(); g.manual_seed(seed)
    ld_tr = DataLoader(ds_tr, batch_size=cfg["bs"], shuffle=True, collate_fn=coll, generator=g)
    ld_va = DataLoader(ds_va, batch_size=cfg["bs"], shuffle=False, collate_fn=coll)

    model = create_model(cfg["model"], cfg["freeze"]).to(DEVICE)
    n_train, n_total = count_params(model)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=cfg["lr"], weight_decay=0.01)
    loss_fn = make_loss(cfg["pos_weight"], y_tr)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    hist, best = [], {"macro_f1": -1}
    t_start = time.time()
    for ep in range(1, cfg["epochs"] + 1):
        model.train(); tl = 0.0
        for ids, att, y in ld_tr:
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                logits = model(input_ids=ids.to(DEVICE), attention_mask=att.to(DEVICE)).logits
            loss = loss_fn(logits.float(), y.to(DEVICE))
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tl += loss.item()
        probs, ytrue = predict_probs(model, ld_va)
        m = evaluate_multi_label(ytrue, probs, 0.5)
        val_loss = float(torch.nn.functional.binary_cross_entropy(torch.tensor(probs).clamp(1e-6, 1 - 1e-6), torch.tensor(ytrue)))
        hist.append({"epoch": ep, "train_loss": tl / len(ld_tr), "val_bce": val_loss, "val_macro_f1": m["macro_f1"]})
        if m["macro_f1"] > best["macro_f1"]:
            best = {"macro_f1": m["macro_f1"], "probs": probs, "metrics": m, "epoch": ep}
            if keep_state:
                best["state"] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    elapsed = time.time() - t_start

    thr, f1_thr = best_global_threshold(ytrue, best["probs"])
    m = best["metrics"]
    row = {"name": cfg.get("name", "run"), "group": cfg.get("group", ""), "seed": seed, "model": cfg["model"],
           "max_len": cfg["max_len"], "trunc": cfg["trunc"], "pos_weight": cfg["pos_weight"], "freeze": cfg["freeze"],
           "lr": cfg["lr"], "epochs": cfg["epochs"], "best_epoch": best["epoch"],
           "macro_f1@0.5": m["macro_f1"], "micro_f1@0.5": m["micro_f1"], "macro_r@0.5": m["macro_r"], "macro_p@0.5": m["macro_p"],
           "best_thr": thr, "macro_f1@best_thr": f1_thr, "hamming@0.5": m["hamming"],
           "trainable_params_M": n_train / 1e6, "time_s": elapsed, "diagnosis": (
               "overfit?" if len(hist) > 1 and hist[-1]["val_bce"] > hist[0]["val_bce"] + 0.02 else "ok")}
    for c in LABEL_COLS:
        row[f"f1_{c}"] = m["per_label"][c]["F1"]
        row[f"recall_{c}"] = m["per_label"][c]["R"]
    return row, best["probs"], hist, best.get("state")

print("Fungsi training siap. Konfigurasi dasar:", BASE_CFG)

## 4. Matriks Eksperimen Terkontrol (multi-seed)

Setiap grup mengubah **tepat satu faktor** dari `BASE_CFG`. Hasil disimpan per-run (`results.csv`, `probs/*.npy`) sehingga
eksekusi dapat **dilanjutkan** setelah sesi terputus (run yang sudah ada dilewati).

| Grup | Faktor yang diubah | Varian |
|---|---|---|
| **A_trunc** | strategi truncation | `standard` · `head_tail` · `tail` |
| **B_posweight** | penanganan imbalance | `none` · `sqrt` · `full` |
| **C_freeze** | lapisan dibekukan | 0 · setengah · semua (linear probe) |
| **D_model** | kapasitas model | `distilroberta-base` · `roberta-base` |
| **E_maxlen** | panjang input | 64 · 128 · 256 |

> [!TIP]
> Estimasi waktu (1.000 train, 2 epoch, per run): GPU T4 ≈ 0,5–1 menit untuk distilroberta, ≈ 2 menit untuk roberta-base;
> CPU ≈ 4 menit (distilroberta). Mulai dari `RUN_GROUPS = ["A_trunc"]`, tambah grup bertahap.

In [ ]:
HALF = None  # diisi di bawah setelah jumlah lapisan diketahui
_n_layers = AutoConfig.from_pretrained(MODEL_SMALL).num_hidden_layers if not SMOKE_TEST else 4

EXPERIMENTS = {
    "A_trunc": [dict(name="A1_standard", trunc="standard"),
                dict(name="A2_head_tail", trunc="head_tail"),
                dict(name="A3_tail_only", trunc="tail")],
    "B_posweight": [dict(name="B1_none", pos_weight="none"),
                    dict(name="B2_sqrt", pos_weight="sqrt"),
                    dict(name="B3_full", pos_weight="full")],
    "C_freeze": [dict(name="C1_freeze0", freeze=0),
                 dict(name=f"C2_freeze{_n_layers // 2}", freeze=_n_layers // 2),
                 dict(name=f"C3_freeze{_n_layers}_probe", freeze=_n_layers)],
    "D_model": [dict(name="D1_distilroberta", model=MODEL_SMALL),
                dict(name="D2_roberta_base", model=MODEL_BASE)],
    "E_maxlen": [dict(name="E1_len64", max_len=64),
                 dict(name="E2_len128", max_len=128),
                 dict(name="E3_len256", max_len=256)],
}

RESULTS_CSV = OUT_DIR / "results.csv"
results = pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else pd.DataFrame()
HISTORIES, STATES = {}, {}

def already_done(name, seed):
    return (not results.empty) and ((results["name"] == name) & (results["seed"] == seed)).any()

plan = [(g, c, s) for g in RUN_GROUPS for c in EXPERIMENTS[g] for s in SEEDS]
print(f"Total run direncanakan: {len(plan)} | sudah selesai (dilewati): {sum(already_done(c['name'], s) for _, c, s in plan)}")

In [ ]:
for g, cfg, seed in plan:
    name = cfg["name"]
    if already_done(name, seed):
        continue
    keep = (seed == SEEDS[0])                     # simpan bobot (memori saja) untuk analisis interpretabilitas
    print(f"\n>>> {g} | {name} | seed={seed}")
    row, probs, hist, state = run_experiment({**cfg, "group": g}, seed, keep_state=keep)
    results = pd.concat([results, pd.DataFrame([row])], ignore_index=True)
    results.to_csv(RESULTS_CSV, index=False)                 # simpan inkremental
    np.save(OUT_DIR / "probs" / f"{name}_s{seed}.npy", probs)
    HISTORIES[(name, seed)] = hist
    if state is not None:
        STATES[(name, seed)] = state
    print(f"    macro-F1@0.5={row['macro_f1@0.5']:.4f} | best_thr={row['best_thr']} -> {row['macro_f1@best_thr']:.4f} | "
          f"{row['time_s']:.0f}s | {row['diagnosis']}")
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

print("\nSelesai. Total baris hasil:", len(results))

## 5. Agregasi: mean ± std dan Uji Berpasangan

Satu run per konfigurasi (seperti eksplorasi sebelumnya) **tidak** dapat membedakan efek nyata dari noise seed.
Di sini dilaporkan mean ± std antar-seed, dan untuk dua konfigurasi dilakukan **bootstrap berpasangan** atas sampel validation.

In [ ]:
def summarize(df):
    cols = ["macro_f1@0.5", "macro_f1@best_thr", "micro_f1@0.5", "macro_r@0.5", "macro_p@0.5", "hamming@0.5", "time_s"]
    g = df.groupby(["group", "name"], sort=False)
    mean, std = g[cols].mean(), g[cols].std(ddof=1).fillna(0)
    out = pd.DataFrame(index=mean.index)
    out["n_seed"] = g.size()
    for c in ["macro_f1@0.5", "macro_f1@best_thr", "macro_r@0.5", "macro_p@0.5"]:
        out[c] = [f"{m:.3f} ± {s:.3f}" for m, s in zip(mean[c], std[c])]
    out["hamming@0.5"] = mean["hamming@0.5"].round(4)
    out["time_s"] = mean["time_s"].round(0)
    return out, mean, std

summary, mean_tbl, std_tbl = summarize(results)
display(summary)

In [ ]:
# F1 per-label (mean antar-seed) -> mana label yang paling terdampak?
per_label = results.groupby(["group", "name"], sort=False)[[f"f1_{c}" for c in LABEL_COLS]].mean()
per_label.columns = LABEL_COLS
display(per_label.round(3))
print(f"Support validation: " + ", ".join(f"{c}={int(df_val[c].sum())}" for c in LABEL_COLS))

fig, ax = plt.subplots(figsize=(10, 4.5))
names = list(mean_tbl.index.get_level_values("name"))
x = np.arange(len(names))
ax.bar(x - 0.2, mean_tbl["macro_f1@0.5"], 0.4, yerr=std_tbl["macro_f1@0.5"], capsize=4, label="Macro-F1 @ τ=0.5")
ax.bar(x + 0.2, mean_tbl["macro_f1@best_thr"], 0.4, yerr=std_tbl["macro_f1@best_thr"], capsize=4, label="Macro-F1 @ τ terbaik (val)")
ax.set_xticks(x); ax.set_xticklabels(names, rotation=30, ha="right")
ax.set_ylabel("Macro-F1 validation"); ax.set_title("Perbandingan konfigurasi (mean ± std antar-seed)")
ax.legend(); plt.tight_layout()
plt.savefig(OUT_DIR / "perbandingan_konfigurasi.png", dpi=200); plt.show()

In [ ]:
def load_probs(name, seeds=None):
    seeds = seeds or sorted(results.loc[results["name"] == name, "seed"].unique())
    return np.stack([np.load(OUT_DIR / "probs" / f"{name}_s{s}.npy") for s in seeds])   # (S, N, 6)

Y_VAL = df_val[LABEL_COLS].values

def bootstrap_diff(name_a, name_b, thr=0.5, n_boot=1000, seed=0):
    """Selisih Macro-F1 (B - A) pada probabilitas rata-rata antar-seed, resampling sampel validation."""
    pa, pb = load_probs(name_a).mean(0), load_probs(name_b).mean(0)
    rng = np.random.RandomState(seed); n = len(Y_VAL); diffs = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        fa = f1_score(Y_VAL[idx], (pa[idx] >= thr).astype(int), average="macro", zero_division=0)
        fb = f1_score(Y_VAL[idx], (pb[idx] >= thr).astype(int), average="macro", zero_division=0)
        diffs.append(fb - fa)
    diffs = np.array(diffs)
    return diffs.mean(), np.percentile(diffs, 2.5), np.percentile(diffs, 97.5)

def compare(name_a, name_b, thr=0.5):
    m, lo, hi = bootstrap_diff(name_a, name_b, thr)
    verdict = "TIDAK meyakinkan (CI mencakup 0)" if lo <= 0 <= hi else ("B lebih baik" if m > 0 else "A lebih baik")
    print(f"{name_b} - {name_a} @ τ={thr}: Δ Macro-F1 = {m:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]  -> {verdict}")

# Contoh: uji H1 (jalankan setelah grup A selesai)
if {"A1_standard", "A2_head_tail"} <= set(results["name"]):
    compare("A1_standard", "A2_head_tail", 0.5)
    compare("A1_standard", "A2_head_tail", 0.4)
if {"B1_none", "B2_sqrt"} <= set(results["name"]):
    compare("B1_none", "B2_sqrt", 0.5)

## 6. Analisis Panjang Teks: Standar vs Head+Tail (uji H1)

Hipotesis dari error analysis: *False Negative* terkonsentrasi pada komentar panjang karena ujaran toksik di bagian akhir terpotong.
Di sini diukur **recall "ada label toksik"** menurut panjang komentar (jumlah token BPE sebelum dipotong), per strategi truncation.

> [!WARNING]
> Jumlah komentar toksik **panjang** pada 300 sampel validation sangat sedikit (puluhan bahkan satu digit). Kolom `n_pos` ditampilkan agar kesimpulan tidak dibaca berlebihan.

In [ ]:
tok_len = get_tokenizer(MODEL_SMALL)
val_len = np.array([len(x) for x in raw_ids(tok_len, df_val["comment_text"], "val")])
y_any = Y_VAL.max(1) == 1
max_inner = BASE_CFG["max_len"] - 2

print(f"Panjang token val: median={np.median(val_len):.0f}, p90={np.percentile(val_len, 90):.0f}, "
      f"terpotong pada max_len={BASE_CFG['max_len']}: {(val_len > max_inner).mean():.1%}")

bins = [("pendek (<=64)", val_len <= 64), (f"sedang (65..{max_inner})", (val_len > 64) & (val_len <= max_inner)),
        (f"panjang (>{max_inner})", val_len > max_inner)]

def recall_by_length(name, thr=0.5):
    p = load_probs(name).mean(0)
    pred_any = (p >= thr).any(1)
    row = {"name": name}
    for label, mask in bins:
        pos = mask & y_any
        row[f"{label} n_pos"] = int(pos.sum())
        row[f"{label} recall"] = round(float(pred_any[pos].mean()), 3) if pos.sum() else np.nan
    row["FPR komentar bersih"] = round(float(pred_any[~y_any].mean()), 3)
    return row

trunc_names = [n for n in ["A1_standard", "A2_head_tail", "A3_tail_only"] if n in set(results["name"])]
if trunc_names:
    for thr in (0.5, 0.3):
        print(f"\n--- τ = {thr} (probabilitas dirata-rata antar-seed) ---")
        display(pd.DataFrame([recall_by_length(n, thr) for n in trunc_names]).set_index("name"))
else:
    print("Jalankan grup A_trunc terlebih dahulu.")

In [ ]:
# Distribusi panjang komentar toksik vs bersih -> seberapa sering sinyal toksik "berada di ekor"?
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].hist([val_len[~y_any].clip(max=600), val_len[y_any].clip(max=600)], bins=30, label=["bersih", "toksik"], density=True)
ax[0].axvline(max_inner, color="red", ls="--", label=f"batas {max_inner}")
ax[0].set_title("Distribusi panjang token (dipotong 600)"); ax[0].legend()
for k, c in enumerate(["toxic", "insult", "obscene"]):
    ax[1].bar(k, (val_len[Y_VAL[:, LABEL_COLS.index(c)] == 1] > max_inner).mean() * 100)
ax[1].set_xticks(range(3)); ax[1].set_xticklabels(["toxic", "insult", "obscene"])
ax[1].set_ylabel("% komentar positif yang terpotong"); ax[1].set_title("Risiko truncation per label")
plt.tight_layout(); plt.savefig(OUT_DIR / "analisis_panjang.png", dpi=200); plt.show()

## 7. Interpretabilitas: *Mengapa* Model Memutuskan Demikian

Dua alat ringan (tanpa pustaka tambahan):

1. **Gradient × input** — kontribusi tiap token terhadap logit label tertentu (`|∂logit/∂emb · emb|`, dijumlah per token, dinormalkan).
2. **Perhatian `<s>` lapisan terakhir** — seberapa besar token `<s>` (yang dipakai head klasifikasi) memperhatikan tiap posisi.

Keduanya dipakai untuk melihat **apakah posisi akhir teks mendapat bobot** pada model standar vs Head+Tail.

> [!NOTE]
> Attribution adalah *indikasi*, bukan penjelasan kausal. Gunakan untuk membangun hipotesis, bukan membuktikannya.
> Perlu bobot hasil training pada sesi yang sama (`STATES`, seed pertama). Jika sesi baru dan run dilewati, jalankan ulang grup terkait dengan menghapus baris di `results.csv`.

In [ ]:
PROFANITY = re.compile(r"\b(fuck\w*|shit\w*|bitch\w*|asshole\w*|dick\w*|cunt\w*|nigg\w*|fag\w*|bastard\w*|idiot\w*|stupid|moron\w*|damn\w*|crap\w*|piss\w*|suck\w*)\b", re.I)

def censor(token):
    t = token.replace("Ġ", " ")
    return PROFANITY.sub(lambda m: m.group(0)[0] + "*" * (len(m.group(0)) - 1), t) if not SHOW_RAW_TEXT else t

def encode_one(tok, text, max_len, mode):
    ids = add_special(tok, truncate_ids(tok(str(text), add_special_tokens=False)["input_ids"], max_len, mode))
    return torch.tensor([ids], dtype=torch.long)

def attributions(model, tok, text, label, max_len=128, mode="standard"):
    model.eval()
    ids = encode_one(tok, text, max_len, mode).to(DEVICE)
    emb = model.get_input_embeddings()(ids).detach().requires_grad_(True)
    out = model(inputs_embeds=emb, attention_mask=torch.ones_like(ids), output_attentions=True)
    logit = out.logits[0, LABEL_COLS.index(label)]
    logit.backward()
    gxi = (emb.grad * emb).sum(-1).abs()[0].detach().cpu().numpy()
    gxi = gxi / (gxi.max() + 1e-12)
    cls_att = out.attentions[-1][0].mean(0)[0].detach().cpu().numpy()          # baris <s>, rata-rata head
    return tok.convert_ids_to_tokens(ids[0].cpu()), gxi, cls_att, torch.sigmoid(logit).item()

def pick_examples(k=3):
    """Komentar toksik PANJANG (terpotong) pada validation."""
    idx = np.where(y_any & (val_len > max_inner))[0]
    return idx[:k]

def load_state_model(name, seed=None):
    seed = seed or SEEDS[0]
    if (name, seed) not in STATES:
        return None
    m = create_model(BASE_CFG["model"], eager_attn=True)
    m.load_state_dict(STATES[(name, seed)]); return m.to(DEVICE)

ex_idx = pick_examples(3)
print("Contoh komentar toksik panjang pada validation (indeks):", list(ex_idx))

In [ ]:
tok = get_tokenizer(BASE_CFG["model"])
compare_cfg = [("A1_standard", "standard"), ("A2_head_tail", "head_tail")]
models = {n: load_state_model(n) for n, _ in compare_cfg}

if any(m is None for m in models.values()):
    print("Bobot belum tersedia di memori (STATES). Jalankan grup A_trunc pada sesi ini (hapus baris A1/A2 dari results.csv jika perlu).")
elif len(ex_idx) == 0:
    print("Tidak ada komentar toksik panjang pada sampel validation ini. Naikkan N_VAL atau ganti seed.")
else:
    pos_curves = {n: [] for n, _ in compare_cfg}
    for i in ex_idx:
        text = df_val.loc[i, "comment_text"]
        label = next(c for c in ["toxic", "insult", "obscene"] if df_val.loc[i, c] == 1) if df_val.loc[i, ["toxic", "insult", "obscene"]].sum() else "toxic"
        fig, axes = plt.subplots(1, len(compare_cfg), figsize=(12, 3.4), sharey=True)
        for ax, (n, mode) in zip(axes, compare_cfg):
            toks, gxi, att, p = attributions(models[n], tok, text, label, mode=mode)
            ax.plot(gxi, label="gradient×input"); ax.plot(att / (att.max() + 1e-12), label="attention <s>", alpha=0.7)
            if mode == "head_tail": ax.axvline(len(toks) // 2, color="gray", ls=":", label="sambungan head|tail")
            ax.set_title(f"{n} | P({label})={p:.2f} | {len(toks)} token"); ax.set_xlabel("posisi token")
            top = np.argsort(-gxi)[:6]
            print(f"[{n}] contoh {i} label={label}: token paling berpengaruh -> {[censor(toks[t]) for t in top]}")
            pos_curves[n].append(np.interp(np.linspace(0, 1, 20), np.linspace(0, 1, len(gxi)), gxi))
        axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()

    plt.figure(figsize=(6.5, 3.6))
    for n, curves in pos_curves.items():
        plt.plot(np.linspace(0, 1, 20), np.mean(curves, 0), marker="o", label=n)
    plt.xlabel("posisi relatif dalam input (0=awal, 1=akhir)"); plt.ylabel("rata-rata gradient×input")
    plt.title("Seberapa besar bagian akhir teks berpengaruh?"); plt.legend(); plt.tight_layout()
    plt.savefig(OUT_DIR / "posisi_atribusi.png", dpi=200); plt.show()
    del models

## 8. Kalibrasi Probabilitas (Temperature Scaling)

Threshold hanya bermakna jika probabilitas **terkalibrasi**. `pos_weight` hampir pasti menggeser probabilitas ke atas.
Diukur **ECE** (Expected Calibration Error) sebelum & sesudah *temperature scaling* (satu parameter `T` pada logit).

> [!WARNING]
> `T` dicocokkan pada validation yang sama dengan evaluasi sehingga ECE "sesudah" bersifat **optimistis** (eksploratif).
> Untuk klaim final, pisahkan validation menjadi dua bagian (fit `T` vs evaluasi).

In [ ]:
from scipy.optimize import minimize_scalar

def to_logit(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))

def ece(y, p, n_bins=10):
    y, p = y.ravel(), p.ravel(); bins = np.linspace(0, 1, n_bins + 1); e = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (p > lo) & (p <= hi)
        if m.any(): e += m.mean() * abs(y[m].mean() - p[m].mean())
    return e

def fit_temperature(y, p):
    z = to_logit(p)
    def nll(T):
        q = np.clip(1 / (1 + np.exp(-z / T)), 1e-6, 1 - 1e-6)
        return -np.mean(y * np.log(q) + (1 - y) * np.log(1 - q))
    return minimize_scalar(nll, bounds=(0.2, 10), method="bounded").x

rows = []
for name in results["name"].unique():
    p = load_probs(name).mean(0)
    T = fit_temperature(Y_VAL, p)
    p_cal = 1 / (1 + np.exp(-to_logit(p) / T))
    rows.append({"name": name, "T": round(T, 2), "ECE sebelum": round(ece(Y_VAL, p), 4), "ECE sesudah": round(ece(Y_VAL, p_cal), 4),
                 "mean prob (prediksi)": round(p.mean(), 3), "rate positif sebenarnya": round(Y_VAL.mean(), 3)})
display(pd.DataFrame(rows).set_index("name"))
print("Baca: T>1 = model terlalu percaya diri; mean prob >> rate positif sebenarnya = bias ke atas (efek pos_weight).")

## 9. Penyimpanan Hasil & Template Kesimpulan

In [ ]:
summary_json = {
    "seeds": SEEDS, "n_train": N_TRAIN, "n_val": N_VAL, "epochs": EPOCHS, "base_cfg": BASE_CFG,
    "groups_run": RUN_GROUPS, "smoke_test": SMOKE_TEST,
    "val_support": {c: int(df_val[c].sum()) for c in LABEL_COLS},
    "mean_macro_f1@0.5": {n: round(v, 4) for n, v in results.groupby("name")["macro_f1@0.5"].mean().items()},
    "std_macro_f1@0.5": {n: round(v, 4) for n, v in results.groupby("name")["macro_f1@0.5"].std(ddof=1).fillna(0).items()},
}
with open(OUT_DIR / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_json, f, indent=2, ensure_ascii=False)
print("Tersimpan di:", OUT_DIR)
for p in sorted(OUT_DIR.glob("*")):
    print(" -", p.name)

### Template kesimpulan (isi setelah eksekusi penuh)

| Hipotesis | Hasil (mean ± std) | Bukti statistik | Diterima? | Catatan / keterbatasan |
|---|---|---|---|---|
| H1 Head+Tail | … | CI bootstrap: … | … | n komentar toksik panjang = … |
| H2 `pos_weight` efek terbesar | … | … | … | … |
| H3 Freeze lapisan | … | … | … | … |
| H4 Model besar bantu label langka | … | … | … | support label langka = … |
| H5 `max_len` | … | … | … | % truncation = … |

**Cara melaporkan secara jujur**
1. Sebutkan ukuran sampel (1.000/300) dan jumlah seed pada setiap tabel.
2. Laporkan mean ± std, bukan angka seed terbaik.
3. Bila CI mencakup 0, tulis *"tidak ada bukti yang meyakinkan"*, bukan *"tidak berpengaruh"*.
4. Konfigurasi terbaik **dibekukan**, lalu diuji **satu kali** di test lewat `src/evaluate_test.py`.
5. Cantumkan penggunaan AI sesuai aturan mata kuliah.